# 5G Network Intelligence — MAE-Aligned Forecasting

## Research question

The previous BCE-plus-pinball hurdle lost to matched direct regression at all six development points. Does optimizing the final point forecast rather than only its distributional surrogate improve MAE? This hypothesis is unproven. Keep encoder, head shapes, seed and temporal boundaries unchanged.


## Temporal protocol

Fit/stop/score: 335/383/443 and 395/443/503. All four original KPIs and all original beams. Lookback 168 hours; horizons +1/+6/+24h. Validation 504–671 and holdout are excluded, including from the cloud bundle. Reused development data cannot establish fresh-test generalization.


In [ ]:
# Run All in this review notebook never fits or calibrates.
RUN_TRAINING = False


## Configuration

Two new modes, four hosted GPU fits. Reuse verified historical direct and surrogate-hurdle controls without retraining. Fixed hidden width 64, dropout 0.1, seed 17, batch 1024, maximum 40 epochs and patience 5. No coefficient grid or scoring-window tuning.


In [ ]:
from pathlib import Path
import yaml
import pandas as pd
from IPython.display import display
PROJECT = Path.cwd()
config = PROJECT / 'configs/mae_aligned.yaml'
if config.exists():
    protocol = yaml.safe_load(config.read_text(encoding='utf-8'))
    display(pd.DataFrame(protocol['folds']))


## Preprocessing and model

Reuse global per-KPI fit-prefix imputation/scaling, observed-zero preservation, missing masks and phase4. The shared encoder receives centered history, last scaled levels and causal seasonal means. No fitted per-beam summary enters. Architecture: `src/models/mae_aligned.py`.


## Objectives

**direct_clipped:** apply the physical nonnegative boundary before computing both training and stopping L1, matching deployment projection. This can create flat gradients below zero; no gain is assumed.

**hurdle_mae:** masked scaled mixture-median L1 + 0.1 × (finite-label BCE + positive-only mean pinball). The auxiliary signal remains when p≤0.5 makes the point median constant zero. Quantiles stay noncrossing, with fixed levels 0.05/0.10/0.25/0.50. The auxiliary coefficient is fixed before fitting. Joint point optimization means probability/quantile calibration must not be claimed.


## Training and debugging

Hosted Colab/Kaggle CUDA only. Checkpoints remain on Drive or Kaggle Output. Stopping always uses point MAE in scaled target units. Inspect finite objectives, observed counts and completed patience. An epoch cap is not completed stopping. Historical execution snapshots stay immutable.


In [ ]:
if RUN_TRAINING:
    from scripts.train_mae_aligned import main
    main()
else:
    print('Review mode: fitting skipped.')


## Evaluation

Report original DLPRB-unit MAE, WAPE = sum absolute error / sum absolute actual, full coverage and zero false-positive rate for each fold/horizon. Compare the two new modes with the corresponding saved historical head, MLP-B and LightGBM hurdle. Require independent checksum, temporal-axis and history-only CPU/GPU parity verification before interpreting results.


In [ ]:
# Display available completed results; never invent missing metrics.
for mode in ('direct_clipped', 'hurdle_mae'):
    path = PROJECT / f'reports/mae_aligned_{mode}_development.csv'
    if path.exists():
        print(mode)
        display(pd.read_csv(path))
    else:
        print(mode, ': no verified result available')


## Limitations

One seed, reused development windows and piecewise gradients. No architecture search, independent-test superiority or WAPE <10% claim. Negative results must remain visible. Do not refit on final validation or promote a model solely because the new objective sounds more aligned.


## Completed and verified result

Four Kaggle T4 fits completed patience. The mixture-MAE objective improves the old surrogate hurdle at all six points but remains weaker than strong historical controls. Clipped direct improves zero false positives at every point, with mixed MAE effects. No standalone promotion or validation refit. Full evidence: `reports/mae_aligned_analysis.md`.


In [ ]:
# Read existing comparisons; never repeat fitting for presentation.
path = PROJECT / 'reports/mae_aligned_control_comparison.csv'
if path.exists():
    display(pd.read_csv(path))
